# London Airbnb Analysis — Data Wrangling

This notebook loads the raw Airbnb data from Inside Airbnb (London, September 2025), explores its structure, identifies missing values and cleans it ready for analysis.

**Dataset:**
- `listings.csv.gz` — 96,871 listings with 79 columns
- `reviews.csv.gz` — 2,097,996 reviews with comment text
- `neighbourhoods.csv` — 33 London boroughs
- `neighbourhoods.geojson` — geographic boundaries for mapping

## 1. Import Libraries

In [1]:
# Import the libraries we need
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

print("Libraries loaded!")

Libraries loaded!


## 2. Load the Data

In [2]:
# Load the listings data
listings = pd.read_csv('../data/listings.csv.gz', compression='gzip', engine='python')

print("Listings loaded!")
print("Rows:", listings.shape[0])
print("Columns:", listings.shape[1])

Listings loaded!
Rows: 96871
Columns: 79


In [3]:
# Load the reviews data
reviews = pd.read_csv('../data/reviews.csv.gz', compression='gzip', engine='python')

print("Reviews loaded!")
print("Rows:", reviews.shape[0])
print("Columns:", reviews.shape[1])

Reviews loaded!
Rows: 2097996
Columns: 6


In [4]:
# Load the neighbourhoods data
neighbourhoods = pd.read_csv('../data/neighbourhoods.csv')

print("Neighbourhoods loaded!")
print("Rows:", neighbourhoods.shape[0])
print("Columns:", neighbourhoods.shape[1])

Neighbourhoods loaded!
Rows: 33
Columns: 2


In [5]:
# Load the geojson file for mapping
with open('../data/neighbourhoods.geojson') as f:
    geo = json.load(f)

print("GeoJSON loaded!")
print("Number of neighbourhood boundaries:", len(geo['features']))

GeoJSON loaded!
Number of neighbourhood boundaries: 33


## 3. Explore the Data

Before cleaning, we take a quick look at the structure of each dataset.

In [6]:
# Preview the listings data
print("First 3 listings:")
listings.head(3)

First 3 listings:


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,13913,https://www.airbnb.com/rooms/13913,20250914034649,2025-09-16,city scrape,Holiday London DB Room Let-on going,My bright double bedroom with a large window h...,Finsbury Park is a friendly melting pot commun...,https://a0.muscache.com/pictures/miso/Hosting-...,54730,...,4.87,4.78,4.78,NaN,f,2,1,1,0,0.30
1,15400,https://www.airbnb.com/rooms/15400,20250914034649,2025-09-16,city scrape,Bright Chelsea Apartment. Chelsea!,Lots of windows and light. St Luke's Gardens ...,It is Chelsea.,https://a0.muscache.com/pictures/428392/462d26...,60302,...,4.84,4.93,4.74,NaN,f,1,1,0,0,0.51
2,17402,https://www.airbnb.com/rooms/17402,20250914034649,2025-09-16,city scrape,Very Central Modern 3-Bed/2 Bath By Oxford St W1,"You'll have a great time in this beautiful, cl...","Fitzrovia is a very desirable trendy, arty and...",https://a0.muscache.com/pictures/39d5309d-fba7...,67564,...,4.72,4.89,4.61,NaN,f,2,2,0,0,0.32


In [7]:
# Preview the reviews data
print("First 3 reviews:")
reviews.head(3)

First 3 reviews:


,listing_id,id,date,reviewer_id,reviewer_name,comments
0,13913,80770,2010-08-18,177109,Michael,My girlfriend and I hadn't known Alina before ...
1,13913,367568,2011-07-11,19835707,Mathias,Alina was a really good host. The flat is clea...
2,13913,529579,2011-09-13,1110304,Kristin,Alina is an amazing host. She made me feel rig...


In [8]:
# Check what columns we have in listings
print("Listings columns:")
print(listings.columns.tolist())

Listings columns:
['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name', 'description', 'neighborhood_overview', 'picture_url', 'host_id', 'host_url', 'host_name', 'host_since', 'host_location', 'host_about', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_thumbnail_url', 'host_picture_url', 'host_neighbourhood', 'host_listings_count', 'host_total_listings_count', 'host_verifications', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms', 'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price', 'minimum_nights', 'maximum_nights', 'minimum_minimum_nights', 'maximum_minimum_nights', 'minimum_maximum_nights', 'maximum_maximum_nights', 'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm', 'calendar_updated', 'has_availability', 'availability_30', 'availability_60', 'availabili

In [9]:
# Check what columns we have in reviews
print("Reviews columns:")
print(reviews.columns.tolist())

Reviews columns:
['listing_id', 'id', 'date', 'reviewer_id', 'reviewer_name', 'comments']


## 4. Check Missing Values

Before cleaning, we need to understand which columns have missing data and how much. This helps us decide what to drop and what to fill.

In [10]:
# Count missing values in each column
missing_count = listings.isnull().sum()

print("Missing value counts:")
print(missing_count[missing_count > 0])

Missing value counts:
description                      2450
neighborhood_overview           55663
picture_url                         6
host_name                          43
host_since                         41
host_location                   23771
host_about                      47038
host_response_time              31707
host_response_rate              31707
host_acceptance_rate            27760
host_is_superhost                1766
host_thumbnail_url                 41
host_picture_url                   41
host_neighbourhood              51021
host_listings_count                41
host_total_listings_count          41
host_verifications                 41
host_has_profile_pic               41
host_identity_verified             41
neighbourhood                   55662
neighbourhood_group_cleansed    96871
bathrooms                       34846
bathrooms_text                    153
bedrooms                        12775
beds                            34920
price                       

In [11]:
# Check missing values in reviews
print("Reviews missing values:")
print(reviews.isnull().sum())

Reviews missing values:
listing_id         0
id                 0
date               0
reviewer_id        0
reviewer_name      2
comments         201
dtype: int64


In [12]:
# Calculate what percentage of each column is missing
missing_pct = (missing_count / len(listings) * 100).round(1)

print("Missing value percentages:")
print(missing_pct[missing_pct > 0])

Missing value percentages:
description                       2.5
neighborhood_overview            57.5
host_location                    24.5
host_about                       48.6
host_response_time               32.7
host_response_rate               32.7
host_acceptance_rate             28.7
host_is_superhost                 1.8
host_neighbourhood               52.7
neighbourhood                    57.5
neighbourhood_group_cleansed    100.0
bathrooms                        36.0
bathrooms_text                    0.2
bedrooms                         13.2
beds                             36.0
price                            36.0
calendar_updated                100.0
has_availability                  4.4
estimated_revenue_l365d          36.0
first_review                     24.9
last_review                      24.9
review_scores_rating             24.9
review_scores_accuracy           24.9
review_scores_cleanliness        24.9
review_scores_checkin            24.9
review_scores_communica

In [13]:
# Combine count and percentage into one table
missing_df = pd.DataFrame({
    'missing_count': missing_count,
    'missing_%': missing_pct
})

# Only show columns that have missing values, worst first
missing_df = missing_df[missing_df['missing_count'] > 0]
missing_df = missing_df.sort_values('missing_%', ascending=False)

print("Listings missing values summary:")
print(missing_df.head(20))

Listings missing values summary:
                              missing_count  missing_%
license                               96871      100.0
neighbourhood_group_cleansed          96871      100.0
calendar_updated                      96871      100.0
neighborhood_overview                 55663       57.5
neighbourhood                         55662       57.5
host_neighbourhood                    51021       52.7
host_about                            47038       48.6
price                                 34908       36.0
beds                                  34920       36.0
bathrooms                             34846       36.0
estimated_revenue_l365d               34908       36.0
host_response_time                    31707       32.7
host_response_rate                    31707       32.7
host_acceptance_rate                  27760       28.7
last_review                           24122       24.9
first_review                          24122       24.9
reviews_per_month               

In [14]:
# Check missing values in reviews
print("Reviews missing values:")
print(reviews.isnull().sum())

Reviews missing values:
listing_id         0
id                 0
date               0
reviewer_id        0
reviewer_name      2
comments         201
dtype: int64


## 5. Clean the Data

Based on the missing values analysis:
- Drop columns that are 100% empty — `license`, `neighbourhood_group_cleansed`, `calendar_updated`, `neighbourhood`, `host_neighbourhood`, `neighborhood_overview`
- Clean the `price` column by removing the dollar sign and converting to a number
- Remove rows where `price`, `beds` or `bathrooms` are missing
- Remove listings with a price of £0
- Remove reviews with no comment text

In [15]:
# Drop columns that are completely empty - no point keeping them
listings = listings.drop(columns=[
    'license',
    'neighbourhood_group_cleansed',
    'calendar_updated',
    'neighbourhood',
    'host_neighbourhood',
    'neighborhood_overview'
])

print("Columns dropped!")
print("Columns remaining:", len(listings.columns))

Columns dropped!
Columns remaining: 73


In [16]:
# Remove the dollar sign from price
listings['price'] = listings['price'].str.replace('$', '', regex=False)

print("Dollar sign removed!")
print(listings['price'].head())

Dollar sign removed!
0     70.00
1    149.00
2    411.00
3       NaN
4    210.00
Name: price, dtype: object


In [17]:
# Remove commas from price
listings['price'] = listings['price'].str.replace(',', '', regex=False)

print("Commas removed!")
print(listings['price'].head())

Commas removed!
0     70.00
1    149.00
2    411.00
3       NaN
4    210.00
Name: price, dtype: object


In [18]:
# Convert price from text to a number
listings['price'] = listings['price'].astype(float)

print("Price converted to number!")
print(listings['price'].head())

Price converted to number!
0     70.0
1    149.0
2    411.0
3      NaN
4    210.0
Name: price, dtype: float64


In [19]:
# Remove rows where price is missing
listings = listings.dropna(subset=['price', 'beds', 'bathrooms'])

print("Rows with missing price/beds/bathrooms removed!")
print("Listings remaining:", len(listings))

Rows with missing price/beds/bathrooms removed!
Listings remaining: 61763


In [20]:
# Remove listings with a price of 0 - these are not valid
listings = listings[listings['price'] > 0]

print("Zero price listings removed!")
print("Listings remaining:", len(listings))

Zero price listings removed!
Listings remaining: 61763


In [21]:
# Remove reviews with no comment text
reviews = reviews.dropna(subset=['comments'])

print("Empty reviews removed!")
print("Reviews remaining:", len(reviews))

Empty reviews removed!
Reviews remaining: 2097795


## 6. Summary

After cleaning we have a much more usable dataset. The key findings from the cleaning process:
- Dropped 6 columns that were 100% empty
- Removed ~35,000 listings with missing price, beds or bathroom data
- Only 201 reviews had no comment text — minimal data loss
- Price ranges from £7 to £1,085,147 — outliers will be handled in the EDA notebook

In [22]:
# Final summary of cleaned data
print("=== CLEANED DATA SUMMARY ===")
print("Listings:", len(listings), "rows,", len(listings.columns), "columns")
print("Reviews:", len(reviews), "rows,", len(reviews.columns), "columns")
print("")
print("Price stats:")
print("Min price: £", listings['price'].min())
print("Max price: £", listings['price'].max())
print("Average price: £", round(listings['price'].mean(), 2))
print("Median price: £", listings['price'].median())

=== CLEANED DATA SUMMARY ===
Listings: 61763 rows, 73 columns
Reviews: 2097795 rows, 6 columns

Price stats:
Min price: £ 7.0
Max price: £ 1085147.0
Average price: £ 229.93
Median price: £ 135.0


## 7. Save Cleaned Data

Save the cleaned files so we don't have to redo the cleaning in every notebook.

In [23]:
# Save cleaned reviews to a new CSV file
reviews.to_csv('../data/reviews_clean.csv', index=False)

print("Cleaned reviews saved!")

Cleaned reviews saved!


## 8. Export for Tableau
The full `listings_clean.csv` has an `amenities` column that contains commas inside the values, which breaks Tableau's CSV parser. This exports a clean, slim version with just the columns needed for the dashboard.

In [24]:
# Export a Tableau-friendly version of listings with key columns only
# The full listings_clean.csv has an amenities column with commas inside values
# which breaks Tableau's CSV parser — this version is safe to use in Tableau

cols_to_keep = [
    'id', 'name', 'neighbourhood_cleansed', 'latitude', 'longitude',
    'room_type', 'property_type', 'price', 'accommodates', 'bedrooms',
    'beds', 'host_is_superhost', 'review_scores_rating',
    'review_scores_cleanliness', 'review_scores_location',
    'number_of_reviews', 'availability_365', 'instant_bookable',
    'reviews_per_month'
]

listings_tableau = listings[cols_to_keep].copy()
listings_tableau.to_csv('../data/listings_tableau.csv', index=False)
print(f"Saved listings_tableau.csv — {listings_tableau.shape[0]} rows, {listings_tableau.shape[1]} columns")

Saved listings_tableau.csv — 61763 rows, 19 columns
